In [ ]:
#Tiền xử lý - SQLSPARK

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, row_number, monotonically_increasing_id
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import to_date, col

# Khởi tạo SparkSession kết nối với HDFS
spark = SparkSession.builder \
    .appName("Ecommerce_Data_Cleaning") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .getOrCreate()

# Đọc dữ liệu từ HDFS
hdfs_path = "hdfs://localhost:9000/user/bigdata/data/e-commerce_shopper_behaviour_and_lifestyle.csv"
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# Kiểm tra số lượng dòng ban đầu
print(f"Số lượng bản ghi ban đầu: {df.count()}")
df.printSchema()

# ---TIỀN XỬ LÝ---

Số lượng bản ghi ban đầu: 1000000
root
 |-- user_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: integer (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: integer (nullable = true)
 |-- cart_abandonment_rate: integer (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: integer (nullable = true)
 |-- preferred_payment_method: string (nullabl

In [2]:
df_no_dup = df.dropDuplicates(["user_id"])
print(f"Số dòng sau khi xóa trùng lặp: {df_no_dup.count()}")

Số dòng sau khi xóa trùng lặp: 1000000


In [3]:
# Thống kê Null
print("Thống kê Null:")
df_no_dup.select([count(when(col(c).isNull(), c)).alias(c) for c in ["age", "income_level", "gender"]]).show()
# Điền giá trị
stats = df_no_dup.approxQuantile(["income_level", "monthly_spend", "age"], [0.5], 0.01)
income_median, spend_median, age_median = stats[0][0], stats[1][0], stats[2][0]

df_filled = df_no_dup.fillna({
    "income_level": income_median,
    "monthly_spend": spend_median,
    "age": age_median,
    "gender": "Unknown",
    "country": "Unknown",
    "employment_status": "Other"
})

print("Thống kê Null sau khi xử lý:")
df_filled.select([count(when(col(c).isNull(), c)).alias(c) for c in ["age", "income_level", "gender"]]).show()


Thống kê Null:
+---+------------+------+
|age|income_level|gender|
+---+------------+------+
|  0|           0|     0|
+---+------------+------+

Thống kê Null sau khi xử lý:
+---+------------+------+
|age|income_level|gender|
+---+------------+------+
|  0|           0|     0|
+---+------------+------+



In [4]:
# Loại bỏ cột chỉ có 1 giá trị
for column in df_filled.columns:
    if df_filled.select(column).distinct().count() == 1:
        df_filled = df_filled.drop(column)


# Xử lý Outliers
df_no_outlier = df_filled.filter(
    (col("age").between(15, 90)) &           # Tuổi từ 15 đến 90
    (col("income_level") > 0) &             # Thu nhập phải dương
    (col("monthly_spend") >= 0) &           # Chi tiêu không được âm
    (col("household_size") >= 1)            # Gia đình ít nhất 1 người
)


# Ép kiểu dữ liệu
df_final = df_no_outlier \
    .withColumn("last_purchase_date", to_date(col("last_purchase_date"), "yyyy-MM-dd")) \
    .withColumn("age", col("age").cast(IntegerType())) \
    .withColumn("income_level", col("income_level").cast(DoubleType())) \
    .withColumn("monthly_spend", col("monthly_spend").cast(DoubleType()))

# Kiểm tra dataset
print(f"--- Tổng số dòng SAU KHI LÀM SẠCH: {df_final.count()} ---")
print("Bảng thống kê dữ liệu sạch:")
df_final.select("age", "income_level", "monthly_spend").summary().show()


--- Tổng số dòng SAU KHI LÀM SẠCH: 1000000 ---
Bảng thống kê dữ liệu sạch:
+-------+-----------------+-----------------+------------------+
|summary|              age|     income_level|     monthly_spend|
+-------+-----------------+-----------------+------------------+
|  count|          1000000|          1000000|           1000000|
|   mean|        49.003377|    104994.565463|       2498.775654|
| stddev|18.19395873953787|54851.47665219791|1444.2086738790772|
|    min|               18|          10000.0|               0.0|
|    25%|               33|          57459.0|            1249.0|
|    50%|               49|         105003.0|            2497.0|
|    75%|               65|         152486.0|            3750.0|
|    max|               80|         200000.0|            5000.0|
+-------+-----------------+-----------------+------------------+



In [5]:
hdfs_write_path = "hdfs://localhost:9000/user/bigdata/data/cleaned_dataset"
df_final.write.csv(hdfs_write_path, header=True, mode="overwrite")

print(f"Đã lưu dữ liệu sạch vào: {hdfs_write_path}")

Đã lưu dữ liệu sạch vào: hdfs://localhost:9000/user/bigdata/data/cleaned_dataset
